# 금융상품 Agent 원클릭 테스트

아래 설정 셀에서 `QUESTION`만 바꾼 뒤 상단의 **Run All(모두 실행)** 을 한 번 누르세요. 실제 파이프라인을 정확히 1회 실행하고 다음 내용을 한 화면에 순서대로 표시합니다.

1. 최종 답변
2. 원격 Graph 연결 방식과 실행 상태
3. 최초/검수 의도와 실행 계획
4. RDB SQL·Graph SPARQL·Vector 출처 및 반환값
5. 노드·모델·실제 하위 호출 요약과 저장 위치

권장 커널: `C:\Users\admin\.venvs\mirae-agent\Scripts\python.exe`. Run All은 Clova 및 원격 DB를 실제 호출하므로 질문당 한 번만 실행하세요. 키는 기존 `.env`에서 읽으며 노트북에 입력하지 않습니다. 새 소스 반영 후에는 반드시 커널을 재시작하세요.

In [ ]:
# ==================== 여기만 수정 ====================
QUESTION = """캠브리콘이 편입된 중국 반도체 ETF를 알려줘."""
QUESTION_ID = "manual-oneclick"
ROW_LIMIT = 20          # 화면에 보일 반환 행 수. DB 조회 범위와는 무관
ALLOW_REPEAT = False    # 같은 커널·같은 질문의 중복 과금 방지
# ====================================================

from pathlib import Path
from html import escape
from urllib.parse import urlsplit
import difflib
import os
import socket
import sys
from IPython.display import HTML, display

# 어느 폴더에서 커널을 시작해도 이 worktree의 소스를 찾습니다.
bases = list(dict.fromkeys([Path.cwd(), *Path.cwd().parents]))
ROOT = next((candidate for base in bases
             for candidate in (base, base / 'worktrees' / 'T-139-catalog-sql')
             if (candidate / 'test/catalog-sql/manual_trace.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('manual_trace.py가 있는 저장소에서 노트북을 여세요.')
ROOT = ROOT.resolve()
os.chdir(ROOT)
helper_path = str(ROOT / 'test/catalog-sql')
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)
import manual_trace as trace
if trace.ROOT != ROOT:
    raise RuntimeError('다른 worktree의 코드가 로드됐습니다. 커널을 재시작하세요.')

ENV_FILES = trace.discover_env_files(ROOT)
REPORT = globals().get('REPORT')

# 검증기는 네트워크를 차단하고 이 플래그를 자동으로 끕니다. 일반 노트북은
# Run All 즉시 실제 질의를 한 번 실행합니다.
OFFLINE_QA = (os.getenv('MIRAE_NOTEBOOK_OFFLINE_QA') == '1' or
              getattr(socket.create_connection, '__name__', '') == '_forbid_network')
RUN_LIVE = not OFFLINE_QA

print('질문:', QUESTION.strip())
print('Python:', sys.version.split()[0], '| 소스:', ROOT)
print('설정 파일 계층:', [str(path) for path in ENV_FILES])
print('모드:', '오프라인 QA' if OFFLINE_QA else '실제 질의 1회 실행')


In [ ]:
def heading(number, title):
    display(HTML(f"<h2 style='margin-top:28px'>{number}. {escape(title)}</h2>"))

def collapsed(title, value, language='json'):
    body = trace.json_text(value) if language == 'json' else str(value or '')
    display(HTML(
        f"<details><summary><b>{escape(title)}</b></summary>"
        f"<pre style='white-space:pre-wrap;overflow-wrap:anywhere'>{escape(body)}</pre></details>"
    ))

def progress(event):
    display(HTML(f"<div>✓ {escape(str(event['node']))} · {event['received_s']:.2f}초</div>"))

if RUN_LIVE:
    RUN_LIVE = False  # Run All을 연속으로 눌러도 실행 셀 자체는 재호출하지 않음
    REPORT = None
    try:
        REPORT = trace.run_question(
            QUESTION, env_files=ENV_FILES, question_id=QUESTION_ID,
            allow_repeat=ALLOW_REPEAT, on_event=progress,
        )
    except Exception as exc:
        print(trace.Redactor().text(f'{type(exc).__name__}: {exc}'))
elif REPORT is None:
    print('오프라인 QA: 실제 API 호출을 생략했습니다.')

if REPORT is not None:
    answer = REPORT.get('answer') or {}
    if isinstance(answer, str):
        answer = {'answer': answer}

    heading(1, '최종 답변')
    display(HTML(
        "<div style='padding:14px;border:1px solid #aaa;border-radius:8px;white-space:pre-wrap'>"
        + escape(answer.get('answer') or '최종 답변 없음 — 아래 오류와 단계 상태를 확인하세요.')
        + '</div>'
    ))
    collapsed('retrieved_context', answer.get('retrieved_context'))
    collapsed('think_trace', answer.get('think_trace'))

    heading(2, '실행 상태와 Graph 전송 경로')
    trace.show_json({key: REPORT.get(key) for key in
                     ('question_id', 'question', 'status', 'elapsed_s', 'exception', 'run_dir')},
                    '실행 요약')
    try:
        from agent.graph_logic import graph_engine
        client = graph_engine._CLIENT
        endpoint = urlsplit(client.endpoint) if client.endpoint else None
        trace.show_json({
            'mode': ('remote_http_direct' if client.endpoint and
                     not getattr(client, '_local_store_enabled', True) else 'explicit_store_or_local'),
            'local_probe_enabled': getattr(client, '_local_store_enabled', None),
            'remote_store_path': str(client.remote_store_path) if client.remote_store_path else None,
            'endpoint': f'{endpoint.scheme}://{endpoint.hostname}:{endpoint.port}{endpoint.path}' if endpoint else None,
        }, 'Graph 연결 설정')
    except Exception as exc:
        print('Graph 설정 확인 실패:', trace.Redactor().text(exc))
    manifest = REPORT.get('manifest') or {}
    collapsed('코드·데이터 버전', {key: manifest.get(key) for key in
              ('commit', 'dirty', 'python_executable', 'source_fingerprint', 'release_id', 'started_at', 'ended_at')})

    heading(3, '의도 분석과 실행 계획')
    initial = REPORT.get('initial_intent')
    verified = REPORT.get('verified_intent')
    collapsed('최초 의도 분석', initial)
    collapsed('검수 후 의도', verified)
    if initial is not None and verified is not None:
        diff = '\n'.join(difflib.unified_diff(
            trace.json_text(initial).splitlines(), trace.json_text(verified).splitlines(),
            fromfile='initial_intent', tofile='verified_intent', lineterm='',
        ))
        collapsed('의도 변경점', diff or '(변경 없음)', language='diff')
    trace.show_table(REPORT.get('plan') or [], limit=None)
    collapsed('라우팅 상세', REPORT.get('route'))

    heading(4, 'RDB · Graph · Vector 실행 근거')
    step_results = REPORT.get('step_results') or {}
    trace.show_table([
        {'step_id': sid, 'engine': result.get('engine'),
         'status': result.get('status') or ('error' if result.get('error') else 'returned'),
         'rows': result.get('rows_total', len(result.get('rows') or result.get('chunks') or [])),
         'reason': result.get('error') or result.get('skipped_reason') or result.get('note')}
        for sid, result in step_results.items()
    ], limit=None)
    for engine in ('rdb', 'graph', 'vector'):
        display(HTML(f"<h3>{engine.upper()}</h3>"))
        selected = [(sid, result) for sid, result in step_results.items() if result.get('engine') == engine]
        if not selected:
            print('해당 엔진을 사용하지 않았습니다.')
        for sid, result in selected:
            collapsed(f'{sid} 상태·조건', {key: value for key, value in result.items()
                      if key not in {'sql', 'sparql', 'rows', 'chunks', 'evidence', 'hydration_queries'}})
            for kind in ('sql', 'sparql'):
                if result.get(kind):
                    collapsed(f'{sid} {kind.upper()} 전문', result[kind], language=kind)
            for item in result.get('hydration_queries') or []:
                collapsed(f"{sid} 상세조회 SQL ({item.get('domain')})", item.get('sql'), language='sql')
            if engine != 'vector':
                trace.show_table(result.get('rows') or [], limit=ROW_LIMIT)
            if result.get('evidence'):
                collapsed(f'{sid} 관계 근거 전체', result['evidence'])
            if engine == 'vector':
                chunks = result.get('chunks') or []
                trace.show_table([{key: value for key, value in chunk.items() if key != 'chunk_text'}
                                  for chunk in chunks], limit=ROW_LIMIT)
                for index, chunk in enumerate(chunks[:ROW_LIMIT], 1):
                    collapsed(f'{sid} 청크 {index} 본문', chunk.get('chunk_text'), language='text')

    heading(5, '실제 호출 · 노드 · 모델 사용량')
    calls = REPORT.get('calls') or []
    trace.show_table([
        {'id': call.get('call_id'), 'node': call.get('node'), 'name': call.get('name'),
         'kind': 'SQL' if call.get('sql') else 'SPARQL' if call.get('sparql') else call.get('kind'),
         'duration_s': call.get('duration_s'), 'status': call.get('status'), 'error': call.get('error'),
         'result_size': len(call.get('result')) if isinstance(call.get('result'), (list, dict)) else None}
        for call in calls
    ], limit=None)
    for call in calls:
        query = call.get('sql') or call.get('sparql')
        if query:
            kind = 'SQL' if call.get('sql') else 'SPARQL'
            collapsed(f"호출 {call.get('call_id')} · {call.get('node')} · {kind}", query, language=kind.lower())
    node_runs = REPORT.get('node_runs') or REPORT.get('events') or []
    trace.show_table([{key: node.get(key) for key in
                      ('node', 'name', 'start_s', 'end_s', 'duration_s', 'ms', 'status', 'error')}
                     for node in node_runs], limit=None)
    trace.show_table(REPORT.get('llm_calls') or [], limit=None)
    merged_rows = (REPORT.get('final_state') or {}).get('merged_rows') or []
    print(f'합쳐진 검색 결과: {len(merged_rows)}행 (화면에는 최대 {ROW_LIMIT}행)')
    trace.show_table(merged_rows, limit=ROW_LIMIT)

    print('\n실행 기록 폴더:', REPORT.get('run_dir') or REPORT.get('loaded_from'))
